# Data analysis
## Import libraries and load dataset

In [ ]:
import pandas as pd
from pandas import DataFrame
import os
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import emoji
import spacy
import it_core_news_sm

# Check the cwd
os.getcwd()

In [ ]:
# Import data from CSV
data = pd.read_csv("../data/raw/train_it.csv")
data

## Basic stats

Check how many tweets are present in the data set, the distribution of the two classes and other general info.

In [ ]:
# some basic stats
print(len(data))
print(data["label"].value_counts())
print(data.info())

The classes are pretty unbalanced: 807 tweets don't contain reclaimed slurs while 207 does.

Now we compute the average lenght of the tweets and the bios, to see if they'll fit the model constrains later.

In [ ]:
# Print average length of texts in tokens
data["text_length"] = data["text"].apply(lambda x: len(x.split()))
print(data["text_length"].mean())

# Print average length of biographies in tokens (if available, otherwise do not count them)
data["bio_length"] = data["bio"].apply(
    lambda x: len(x.split()) if pd.notnull(x) else None
)
print(data["bio_length"].mean())

## Lexical and Semantic Analysis

### Tokenization and Cleaning

Check for user mentions and url to see if they are useful to keep or not.

In [ ]:
# Check for user mentions
data["user_mention"] = data["text"].apply(lambda x: re.findall(r"\s@\w+", x))

data[data["user_mention"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for urls
data["urls"] = data["text"].apply(lambda x: re.findall(r"url", x))
data[data["urls"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for hashtags
data["hashtags"] = data["text"].apply(lambda x: re.findall(r"#\w+", x))
data[data["hashtags"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for emojis

data["emojis"] = data["text"].apply(lambda x: emoji.distinct_emoji_list(x))

data[data["emojis"].apply(lambda x: len(x) > 0)]

Urls have been replaced with the string "url". Since all user mentions are hidden for privacy, we can remove them without losing meaning.

The next function processes the text by removing user mentions, extra space, and useless punctuation. The texts are also transformed in lower-case for further analysis.

In [ ]:
# Function to clean the tweets
def clean_text(text: str) -> str:
    """
    Cleans the input text by performing the following operations:
    - Removes user mentions (e.g., @username).
    - Removes urls.
    - Removes extra spaces and trims the text.
    - Converts the text to lowercase.

    Args:
        text (str): The input text to be cleaned.

    Returns:
        str: The cleaned text.
    """
    # Remove user mentions
    text = re.sub(r"@\w+", "", text)

    # Remove url
    text = re.sub(r"\burl\b", "", text, flags=re.I)

    # Remove useless extra space
    text = re.sub(r"\s+", " ", text).strip()

    # Transform everything in lower_case
    text_clean = text.lower()

    return text_clean

In [ ]:
# Drop useless columns
data = data.drop(columns=["text_length", "bio_length", "user_mention", "urls"])

In [ ]:
data["text_clean"] = data["text"].apply(lambda x: clean_text(x))

In [ ]:
data

### Frequency Analysis

Now we want to perform frequency analyis. Specifically, we want to find the most frequent words for each class. To do so, we need to tokenize the tweets first. We'll use Spacy.

In [ ]:
nlp = spacy.load("it_core_news_sm")

nlp = it_core_news_sm.load()

In [ ]:
def str_freq_analysis(data: DataFrame, column_name) -> Counter:
    """
    Performs frequency analysis on the text data in the specified column.

    This function tokenizes the text data using the spaCy NLP pipeline and counts the frequency
    of words, excluding punctuation and specific parts of speech such as determiners, adpositions,
    pronouns, auxiliaries, conjunctions, particles, and punctuation.

    Args:
        data (DataFrame): A pandas DataFrame containing the text data.
        column_name (str): The name of the column in the DataFrame containing the text to analyze.

    Returns:
        Counter: A Counter object containing the frequency of words in the specified text column.
    """
    freqs = Counter()

    for doc in nlp.pipe(data[column_name], batch_size=50):
        freqs.update(
            [
                token.text.lower()
                for token in doc
                if not token.is_punct
                and token.pos_
                not in {"DET", "ADP", "PRON", "AUX", "CCONJ", "SCONJ", "PART", "PUNCT"}
            ]
        )

    return freqs

In [ ]:
def list_freq_analysis(data: DataFrame, column_name: str) -> Counter:
    """
    Performs frequency analysis on a column containing lists of items.

    Args:
        data (DataFrame): A pandas DataFrame containing the data.
        column_name (str): The name of the column containing lists of items.

    Returns:
        Counter: A Counter object with the frequency of each item in the lists.
    """
    freqs = Counter()

    # Ensure the column exists
    if column_name not in data.columns:
        raise ValueError(f"Column '{column_name}' does not exist in the DataFrame.")

    # Iterate through the column and update the counter
    for items in data[column_name]:
        if isinstance(items, list):  # Ensure the value is a list
            freqs.update(items)

    return freqs

In [ ]:
def plot_common(ax, freqs: Counter, n_common: int = 10) -> None:
    """
    Plots a horizontal bar chart to visualize the most common items or hashtags or emojis and their relative frequencies.

    Args:
        ax (matplotlib.axes.Axes): The axes object to plot the bar chart on.
        freqs (Counter): A Counter object containing item frequencies.
        n_common (int, optional): The number of most common items to display. Defaults to 10.

    Returns:
        None
    """
    common_items = freqs.most_common(n_common)

    items = []
    rel_frequencies = []
    total_items = freqs.total()

    for word, frequency in common_items:
        items.append(word)
        rel_frequencies.append(frequency / total_items)

    y_pos = np.arange(len(items))

    ax.barh(y_pos, rel_frequencies, color="skyblue")
    ax.set_yticks(y_pos, labels=items)
    ax.invert_yaxis()
    ax.set_xlabel("Relative frequency")

In [ ]:
data_reclamation = data[data["label"] == 1]
data_no_rec = data[data["label"] == 0]

In [ ]:
freqs_hashtags = list_freq_analysis(data, "hashtags")

print(freqs_hashtags.most_common(20))

In [ ]:
plt.rcParams["font.family"] = "Segoe UI Emoji"

fig, axs = plt.subplots(4, 2, figsize=(16, 20))

freqs_all = str_freq_analysis(data, "text_clean")
freqs_rec = str_freq_analysis(data_reclamation, "text_clean")
freqs_no_rec = str_freq_analysis(data_no_rec, "text_clean")
freqs_hashtags_rec = list_freq_analysis(data_reclamation, "hashtags")
freqs_hashtags_no_rec = list_freq_analysis(data_no_rec, "hashtags")
freqs_emojis_rec = list_freq_analysis(data_reclamation, "emojis")
freqs_emojis_no_rec = list_freq_analysis(data_no_rec, "emojis")

plot_common(axs[0, 0], freqs_all, 15)
plot_common(axs[1, 0], freqs_rec, 15)
plot_common(axs[1, 1], freqs_no_rec, 15)
plot_common(axs[2, 0], freqs_hashtags_rec, 15)
plot_common(axs[2, 1], freqs_hashtags_no_rec, 15)
plot_common(axs[3, 0], freqs_emojis_rec, 15)
plot_common(axs[3, 1], freqs_emojis_no_rec, 15)

axs[0, 0].set_title("word frequency for all the tweets")
axs[1, 0].set_title("word frequency for reclamation tweets")
axs[1, 1].set_title("word frequency for non reclamation tweets")
axs[2, 0].set_title("hashtag frequency for reclamation tweets")
axs[2, 1].set_title("hashtag frequency for non reclamation tweets")
axs[3, 0].set_title("emoji frequency for reclamation tweets")
axs[3, 1].set_title("emoji frequency for non reclamation tweets")

plt.show()

The plots give us some insights:
- In the most frequent words of the non-reclamation tweets, there are no slurs. The words *gay* and *trans* can sometimes be used as slurs, but they usually have a neutral meaning.
- In the reclamation tweets, different variations of *frocio* (e.g., frocio, forcio, forci, froci, frocia) are among the most frequent words. Some of these are misspelled, which could pose challenges during the model's training phase.
- Words that suggest reclamation, such as *pride*, are not particularly informative in distinguishing the label of the tweet.
- Some emojis seem to be informative with respect to the tweet label.

### Characteristic Term Analysis

In [ ]:
# Import hurtlex data from TSV
hurtlex = pd.read_csv("../data/ras/hurtlex_IT.tsv", sep="\t")

hurtlex.head()

In [ ]:
homophobic_terms = hurtlex[hurtlex["category"] == "om"]

homophobic_terms

In [ ]:
# create a set of homophobic terms for faster lookup
homophobic_set = set(homophobic_terms["lemma"].str.lower())

# add also feminine forms (simple heuristic: add 'a' at the end if it ends with 'o')
homophobic_set.update(
    {term[:-1] + "a" for term in homophobic_set if term.endswith("o")}
)

print(len(homophobic_set))

In [ ]:
# Check for homophobic terms in the tweets end return a column with the terms found
def contains_homophobic_term(text):
    words = text.lower().split()
    for word in words:
        if word in homophobic_set:
            return word
    return None


data["contains_homophobic"] = data["text"].apply(contains_homophobic_term)
data[["text", "contains_homophobic"]]

In [ ]:
# Drop rows with None values in 'contains_homophobic' column
data_with_homophobic = data.dropna(subset=["contains_homophobic"])

data_with_homophobic[["text", "contains_homophobic"]]

In [ ]:
data_with_homophobic[data_with_homophobic["contains_homophobic"] == "queer"][
    ["text", "contains_homophobic", "label"]
]